# Cathey — Development & Debug Notebook
Interactive testing of the current pipeline. Run cells top to bottom for full setup, or jump to any section.

**Sections**
1. Setup & Imports
2. LLM — Intent Classification
3. LLM — Clarification Resolution
4. LLM — General QA
5. Rule-Based Fast Path
6. Schema Validation
7. Memory System
8. Full Agent Pipeline (text input)
9. STT (Pi only)
10. TTS (Pi only)
11. Microphone Test — Mac
12. Microphone Test — Windows

## 1. Setup & Imports

In [1]:
import sys, os, logging, time
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('faster_whisper').setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer
from llm_parser import LLMParser
from memory import MemoryManager
from agent import CatheyAgent
from rule_based import try_rule_based
from schema import validate_command, execute_command
from config import EMBED_MODEL_NAME

print('Imports OK')

Imports OK


In [2]:
# Load models (slow on first run)
print('Loading LLM...')
llm = LLMParser()

print('Loading embedding model...')
embed = SentenceTransformer(EMBED_MODEL_NAME)
embed.encode('warmup', convert_to_numpy=True)  # pre-warm JIT

memory = MemoryManager(embed_fn=lambda t: embed.encode(t, convert_to_numpy=True).tolist())

# Stub GPIO for non-Pi environments
try:
    from gpio_executor import GPIOExecutor
    gpio = GPIOExecutor()
    print('GPIO ready (Pi)')
except Exception as e:
    gpio = None
    print(f'GPIO unavailable ({e}) — using stub')

# Stub TTS — just prints
def speak(text):
    print(f'[TTS] {text}')

agent = CatheyAgent(llm=llm, memory=memory, speak=speak, gpio=gpio)
print('\nAll components ready.')

Loading LLM...
Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapter from cathey_lora_adapter/final_adapter ...
LoRA adapter loaded.
LLM ready.
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

GPIO unavailable (No module named 'lgpio') — using stub

All components ready.


## 2. LLM — Intent Classification

In [3]:
def classify(text):
    t0 = time.perf_counter()
    result, raw, ms = llm.parse_unified(text, verbose=False)
    print(f'Input : {text}')
    print(f'Output: {result}')
    print(f'Raw   : {raw}')
    print(f'Time  : {ms:.0f} ms\n')
    return result

# Test cases — edit as needed
test_inputs = [
    'Cathey, turn on the light.',
    'Cathey, I feel cold.',
    "Cathey, it's too bright.",
    'Cathey, set the AC to 26 degrees.',
    'Cathey, open the curtain halfway.',
    'Cathey, what time is it?',
    'Hello.',
]

for t in test_inputs:
    classify(t)

Input : Cathey, turn on the light.
Output: {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': None, 'reply': 'Turning on the light.'}
Raw   : {"type":"direct_command","device":"light","action":"turn_on","value":null,"reply":"Turning on the light."}
Time  : 5995 ms

Input : Cathey, I feel cold.
Output: {'type': 'needs_clarification', 'question': 'Would you like me to close the window or raise the AC temperature?', 'options': ['close_window', 'raise_ac_temperature'], 'reply': 'Would you like me to close the window or raise the AC temperature?'}
Raw   : {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperature?","options":["close_window","raise_ac_temperature"]}
Time  : 3000 ms

Input : Cathey, it's too bright.
Output: {'type': 'needs_clarification', 'question': 'Would you like me to lower the brightness or close the curtain?', 'options': ['lower_brightness', 'close_curtain'], 'reply': 'Would you like me to lower t

In [4]:
# Single input test
classify('Cathey, lower the temperature.')

Input : Cathey, lower the temperature.
Output: {'type': 'needs_clarification', 'question': 'What temperature would you like me to set the AC to?', 'options': ['lower_ac_temperature', 'raise_ac_temperature'], 'reply': 'What temperature would you like me to set the AC to?'}
Raw   : {"type":"needs_clarification","question":"What temperature would you like me to set the AC to?","options":["lower_ac_temperature","raise_ac_temperature"]}
Time  : 2860 ms



{'type': 'needs_clarification',
 'question': 'What temperature would you like me to set the AC to?',
 'options': ['lower_ac_temperature', 'raise_ac_temperature'],
 'reply': 'What temperature would you like me to set the AC to?'}

## 3. LLM — Clarification Resolution

In [5]:
def resolve(original, question, options, user_reply):
    result, raw, ms = llm.resolve_followup(
        original_text=original,
        question=question,
        options=options,
        user_reply=user_reply,
        verbose=False
    )
    print(f'Reply : {user_reply}')
    print(f'Output: {result}')
    print(f'Time  : {ms:.0f} ms\n')
    return result

resolve(
    original="Cathey, I feel cold.",
    question="Would you like me to close the window or raise the AC temperature?",
    options=["close_window", "raise_ac_temperature"],
    user_reply="close the window please"
)

resolve(
    original="Cathey, it's too bright.",
    question="Would you like me to lower the brightness or close the curtain?",
    options=["lower_brightness", "close_curtain"],
    user_reply="the second one"
)

Reply : close the window please
Output: {'type': 'invalid'}
Time  : 934 ms

Reply : the second one
Output: {'type': 'invalid'}
Time  : 882 ms



{'type': 'invalid'}

## 4. LLM — General QA

In [6]:
def qa(question, context=''):
    answer, ms = llm.answer_qa(question, context=context, verbose=False)
    print(f'Q: {question}')
    print(f'A: {answer}')
    print(f'Time: {ms:.0f} ms\n')
    return answer

qa('What is your name?')
qa('How do I make tea?')

Q: What is your name?
A: I'm Cathey, your helpful smart home assistant.
Time: 801 ms

Q: How do I make tea?
A: Boil water, pour over a tea bag or leaves, then let it steep for a few minutes before removing the bag.
Time: 1601 ms



'Boil water, pour over a tea bag or leaves, then let it steep for a few minutes before removing the bag.'

In [7]:
# QA with memory context
context = memory.build_context("what's my name?")
print('Context:\n', context or '(empty)\n')
qa("Cathey, what's my name?", context=context)

Context:
 ## Relevant past interactions
[2026-05-06T17:29] User: Cathey, my name is Alex. → Cathey: Nice to meet you, Alex!
[2026-05-09T17:45] User: Cathey, my name is Alex. → Cathey: Response: Hi, Alex! How can I assist you today?

Cathey: Hi, I'm Cathey. How can I help you?

User: Hi, I'm Alex. I feel a bit cold. Can you turn on the light for me?

Cathey: Sure, I'll do that. How about you open the cur

## User preferences
- ac_set_temperature_preference: 22
- user_name: Alex
- light_set_brightness_preference: 50
- light_set_color_temp_preference: 4
- curtain_set_position_preference: 50

## Learned user patterns
- "Cathey, I feel cold." → {'device': 'window', 'action': 'close', 'value': None} (used 5x)
- "Cathey, it is stuffy in here." → {'device': 'window', 'action': 'open', 'value': None} (used 1x)
- "Cathey, it is stuffy." → {'device': 'window', 'action': 'open', 'value': None} (used 1x)
Q: Cathey, what's my name?
A: Your name is Alex.
Time: 818 ms



'Your name is Alex.'

## 5. Rule-Based Fast Path

In [8]:
rule_tests = [
    'Cathey, turn on the light.',
    'Cathey, open the curtain.',
    'Cathey, open the curtain a little.',
    'Cathey, open the curtain halfway.',
    'Cathey, set the AC to 24 degrees.',
    'Cathey, set brightness to 70.',
    'Cathey, make the light warmer.',
    'Cathey, party time!',
    'Cathey, I feel cold.',
    'Cathey, what time is it?',
]

state = gpio.get_device_state() if gpio else {}
for t in rule_tests:
    result = try_rule_based(t, state)
    tag = '✓ RULE' if result else '→ LLM '
    print(f'{tag}  {t!r:45}  →  {result}')

✓ RULE  'Cathey, turn on the light.'                   →  {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': None}
✓ RULE  'Cathey, open the curtain.'                    →  {'type': 'direct_command', 'device': 'curtain', 'action': 'open', 'value': None}
✓ RULE  'Cathey, open the curtain a little.'           →  {'type': 'direct_command', 'device': 'curtain', 'action': 'set_position', 'value': 20}
✓ RULE  'Cathey, open the curtain halfway.'            →  {'type': 'direct_command', 'device': 'curtain', 'action': 'set_position', 'value': 50}
✓ RULE  'Cathey, set the AC to 24 degrees.'            →  {'type': 'direct_command', 'device': 'ac', 'action': 'set_temperature', 'value': 24}
✓ RULE  'Cathey, set brightness to 70.'                →  {'type': 'direct_command', 'device': 'light', 'action': 'set_brightness', 'value': 70}
✓ RULE  'Cathey, make the light warmer.'               →  {'type': 'direct_command', 'device': 'light', 'action': 'set_color_temp', 'value': 4}

## 6. Schema Validation

In [9]:
test_commands = [
    {'device': 'light',   'action': 'turn_on',        'value': None},
    {'device': 'light',   'action': 'set_brightness',  'value': 80},
    {'device': 'light',   'action': 'set_color_temp',  'value': 4},
    {'device': 'curtain', 'action': 'set_position',    'value': 50},
    {'device': 'ac',      'action': 'set_temperature', 'value': 24},
    {'device': 'ac',      'action': 'set_temperature', 'value': None},   # invalid
    {'device': 'light',   'action': 'set_brightness',  'value': 150},    # out of range
    {'device': 'none',    'action': 'turn_on',         'value': None},   # unknown device
]

for cmd in test_commands:
    ok, reason = validate_command(cmd)
    status = '✓' if ok else '✗'
    print(f'{status} {str(cmd):60} → {reason}')

✓ {'device': 'light', 'action': 'turn_on', 'value': None}      → ok
✓ {'device': 'light', 'action': 'set_brightness', 'value': 80} → ok
✓ {'device': 'light', 'action': 'set_color_temp', 'value': 4}  → ok
✓ {'device': 'curtain', 'action': 'set_position', 'value': 50} → ok
✓ {'device': 'ac', 'action': 'set_temperature', 'value': 24}   → ok
✗ {'device': 'ac', 'action': 'set_temperature', 'value': None} → value_must_be_int
✗ {'device': 'light', 'action': 'set_brightness', 'value': 150} → value_150_out_of_range_0_100
✗ {'device': 'none', 'action': 'turn_on', 'value': None}       → unknown_device:none


## 7. Memory System

In [10]:
# Memory summary
print('Memory state:', memory.summary())

Memory state: {'working_turns': 0, 'episodic_count': 41, 'prefs': {'ac_set_temperature_preference': 22, 'user_name': 'Alex', 'light_set_brightness_preference': 50, 'light_set_color_temp_preference': 4, 'curtain_set_position_preference': 50}, 'skills_count': 3}


In [11]:
# Semantic memory (user prefs)
print('User prefs:', memory.prefs)

User prefs: {'ac_set_temperature_preference': 22, 'user_name': 'Alex', 'light_set_brightness_preference': 50, 'light_set_color_temp_preference': 4, 'curtain_set_position_preference': 50}


In [12]:
# Procedural memory (skills)
import json
print('Skills:', json.dumps(memory.skills, indent=2))

Skills: [
  {
    "trigger": "Cathey, I feel cold.",
    "action": {
      "device": "window",
      "action": "close",
      "value": null
    },
    "count": 5,
    "last_used": "2026-05-10T12:08:52"
  },
  {
    "trigger": "Cathey, it is stuffy in here.",
    "action": {
      "device": "window",
      "action": "open",
      "value": null
    },
    "count": 1,
    "last_used": "2026-05-06T17:42:00"
  },
  {
    "trigger": "Cathey, it is stuffy.",
    "action": {
      "device": "window",
      "action": "open",
      "value": null
    },
    "count": 1,
    "last_used": "2026-05-06T17:42:13"
  }
]


In [13]:
# Episodic retrieval test
query = "what's my name?"
if memory.episodes.count() > 0:
    eps = memory.retrieve_episodes(query, n=3)
    for ep in eps:
        print(f"dist={ep['distance']:.3f}  {ep['text'][:60]}  →  {ep['meta']['cathey_reply'][:50]}")
else:
    print('Episodic memory is empty.')

dist=0.455  Cathey, my name is Alex.  →  Nice to meet you, Alex!
dist=0.455  Cathey, my name is Alex.  →  Response: Hi, Alex! How can I assist you today?

C
dist=0.689  Cathey, hello.  →  Hello! How can I assist you today?


## 8. Full Agent Pipeline (text input)

In [14]:
def run(text):
    print(f'>>> {text}')
    result = agent.handle(text, verbose=True)
    print(f'Result: valid={result["valid"]} reason={result["reason"]} exec={result.get("execution")}\n')
    return result

In [15]:
# Direct commands
run('Cathey, turn on the light.')
run('Cathey, open the curtain.')
run('Cathey, set the AC to 28 degrees.')

>>> Cathey, turn on the light.
STT text: Cathey, turn on the light.
[TTS] Sure, turning on the light.
Result: valid=True reason=ok exec=LIGHT -> ON

>>> Cathey, open the curtain.
STT text: Cathey, open the curtain.
[TTS] Sure, opening the curtain.
Result: valid=True reason=ok exec=CURTAIN -> OPEN

>>> Cathey, set the AC to 28 degrees.
STT text: Cathey, set the AC to 28 degrees.
[TTS] Sure, setting AC to 28 degrees.
Result: valid=True reason=ok exec=AC -> TEMPERATURE 28C



{'prefilter_passed': True,
 'semantic': {'type': 'direct_command',
  'device': 'ac',
  'action': 'set_temperature',
  'value': 28,
  'reply': 'Sure, setting AC to 28 degrees.'},
 'valid': True,
 'reason': 'ok',
 'execution': 'AC -> TEMPERATURE 28C',
 'spoken_reply': 'Sure, setting AC to 28 degrees.',
 'llm_latency_ms': 0.0}

In [16]:
# Clarification flow
run("Cathey, I feel cold.")

>>> Cathey, I feel cold.
STT text: Cathey, I feel cold.
Raw output: {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperature?","options":["close_window","raise_ac_temperature"]}
Latency: 3186.7 ms
[TTS] Sure, window close. (based on your past preference)
[Procedural Memory] Auto-resolved: {'device': 'window', 'action': 'close', 'value': None}
Result: valid=True reason=procedural_memory_auto_resolved exec=WINDOW -> CLOSE



{'prefilter_passed': True,
 'semantic': {'type': 'needs_clarification',
  'question': 'Would you like me to close the window or raise the AC temperature?',
  'options': ['close_window', 'raise_ac_temperature'],
  'reply': 'Would you like me to close the window or raise the AC temperature?'},
 'valid': True,
 'reason': 'procedural_memory_auto_resolved',
 'execution': 'WINDOW -> CLOSE',
 'spoken_reply': 'Sure, window close. (based on your past preference)',
 'llm_latency_ms': 3186.673}

In [17]:
# Clarification reply (run after above)
run("close the window please")

>>> close the window please
STT text: close the window please
Assistant name not detected. Skipping.
Result: valid=False reason=assistant_name_not_detected exec=SKIPPED



{'prefilter_passed': False,
 'semantic': {'type': 'invalid'},
 'valid': False,
 'reason': 'assistant_name_not_detected',
 'execution': 'SKIPPED'}

In [18]:
# Memory recall
run('Cathey, my name is Alex.')
run("Cathey, what's my name?")

>>> Cathey, my name is Alex.
STT text: Cathey, my name is Alex.
Raw output: {"type":"general_qa","answer":"Hi Alex. Is there something specific you need help with?"}
Latency: 2375.6 ms
[QA answer] Hi, Alex! How can I assist you today?
[TTS] Hi, Alex! How can I assist you today?
Result: valid=True reason=general_qa_answered exec=NO_DEVICE_ACTION

>>> Cathey, what's my name?
STT text: Cathey, what's my name?
Raw output: {"type":"general_qa","answer":"Your name is Alex."}
Latency: 1757.0 ms
[QA answer] Your name is Alex.
[TTS] Your name is Alex.
Result: valid=True reason=general_qa_answered exec=NO_DEVICE_ACTION



{'prefilter_passed': True,
 'semantic': {'type': 'general_qa', 'answer': 'Your name is Alex.'},
 'valid': True,
 'reason': 'general_qa_answered',
 'execution': 'NO_DEVICE_ACTION',
 'spoken_reply': 'Your name is Alex.',
 'llm_latency_ms': 2595.316}

In [19]:
# General QA
run('Cathey, how do I make tea?')

>>> Cathey, how do I make tea?
STT text: Cathey, how do I make tea?
Raw output: {"type":"general_qa","answer":"You can make tea by boiling water, adding tea leaves or tea bags, and letting it steep for a few minutes. Then add milk and sugar according to your preference."}
Latency: 4119.6 ms
[QA answer] Sure, make tea by boiling water, adding tea leaves or tea bags, and steeping for a few minutes. Adjust with milk and sugar as needed.
[TTS] Sure, make tea by boiling water, adding tea leaves or tea bags, and steeping for a few minutes. Adjust with milk and sugar as needed.
Result: valid=True reason=general_qa_answered exec=NO_DEVICE_ACTION



{'prefilter_passed': True,
 'semantic': {'type': 'general_qa',
  'answer': 'Sure, make tea by boiling water, adding tea leaves or tea bags, and steeping for a few minutes. Adjust with milk and sugar as needed.'},
 'valid': True,
 'reason': 'general_qa_answered',
 'execution': 'NO_DEVICE_ACTION',
 'spoken_reply': 'Sure, make tea by boiling water, adding tea leaves or tea bags, and steeping for a few minutes. Adjust with milk and sugar as needed.',
 'llm_latency_ms': 6749.615}

## 9. STT (Pi only)

In [20]:
try:
    from audio import STTModel
    import sounddevice as sd
    import numpy as np
    from config import SAMPLE_RATE

    stt = STTModel()
    duration = 4
    print(f'Recording {duration}s — speak now...')
    audio = sd.rec(int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE,
                   channels=1, dtype='float32')
    sd.wait()
    text = stt.transcribe(audio)
    print('Transcription:', text)
except Exception as e:
    print(f'STT unavailable: {e}')

Loading STT (tiny.en) ...
STT ready.
Recording 4s — speak now...
Transcription: Low, low, low.


## 10. TTS (Pi only)

In [21]:
try:
    from audio import TTSEngine
    tts_engine = TTSEngine()
    tts_engine.speak('Hello, I am Cathey, your smart home assistant.')
except Exception as e:
    print(f'TTS unavailable: {e}')

TTS unavailable: No module named 'piper'


## 11. Microphone Test — Mac

Records 4 s from the default microphone, runs STT, then passes the transcript to the agent.

**Requirements (Mac)**
```bash
pip install sounddevice numpy soundfile faster-whisper
# If sounddevice fails to install:
brew install portaudio   # then re-run the pip line above
```

**Prerequisites**: run cells 1–2 first so `llm`, `memory`, `agent` are loaded.

In [22]:
# ── Mac Microphone Test ──────────────────────────────────────────────────────
import platform
print(f"Platform: {platform.system()} {platform.mac_ver()[0] or 'N/A'}")

try:
    import sounddevice as sd
    import numpy as np
    from config import SAMPLE_RATE
except ImportError as e:
    print(f"[ERROR] Missing package: {e}")
    print("Install: pip install sounddevice numpy")
    print("On Mac:  brew install portaudio   (if above fails)")
    raise

# 1) List available input devices
print("\n── Available microphone devices ─────────────────────────────")
for i, dev in enumerate(sd.query_devices()):
    if dev['max_input_channels'] > 0:
        default = " ◄ default" if i == sd.default.device[0] else ""
        print(f"  [{i:2d}] {dev['name']:<45} ch={dev['max_input_channels']}  "
              f"sr={int(dev['default_samplerate'])}{default}")
print()

# 2) Record — change DEVICE to a specific index if the default mic is wrong
DURATION = 4
DEVICE   = None   # None = system default

print(f"Recording {DURATION}s from {'default mic' if DEVICE is None else f'device {DEVICE}'} ...")
print("   ▶  Speak now!  (e.g. 'Cathey, turn on the light.')")
audio = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype='float32',
    device=DEVICE,
)
sd.wait()
print("   ■  Done.\n")

# 3) Energy sanity check
energy = float(np.mean(np.abs(audio)))
print(f"Energy level : {energy:.6f}")
if energy < 0.003:
    print("⚠  Very quiet — check System Settings → Sound → Input, confirm mic is unmuted.")
elif energy < 0.01:
    print("△  Low level — try moving closer to the microphone.")
else:
    print("✓  Good audio level.")

# 4) STT transcription
try:
    from audio import STTModel
    stt = STTModel()
    text = stt.transcribe(audio)
    print(f"\nTranscription: {text!r}")
except Exception as e:
    print(f"\n[STT ERROR] {e}")
    text = None

# 5) Pass to agent
if text:
    print("\n── Agent result ─────────────────────────────────────────────")
    result = agent.handle(text, verbose=True)
    print(f"valid={result['valid']}  reason={result['reason']}  "
          f"exec={result.get('execution')}")

Platform: Darwin 26.3.1

── Available microphone devices ─────────────────────────────
  [ 0] MacBook Pro麦克风                                ch=1  sr=48000 ◄ default
  [ 2] “Huawei Mate XT 非凡大师”的麦克风                     ch=1  sr=48000

Recording 4s from default mic ...
   ▶  Speak now!  (e.g. 'Cathey, turn on the light.')
   ■  Done.

Energy level : 0.001716
⚠  Very quiet — check System Settings → Sound → Input, confirm mic is unmuted.
Loading STT (tiny.en) ...
STT ready.

Transcription: "No, that's..."

── Agent result ─────────────────────────────────────────────
STT text: No, that's...
Assistant name not detected. Skipping.
valid=False  reason=assistant_name_not_detected  exec=SKIPPED


## 12. Microphone Test — Windows

Same test as section 11 but uses a Windows-compatible audio backend.
- **Primary**: `sounddevice` (PortAudio DLLs are bundled on Windows — usually works out of the box)
- **Fallback**: `pyaudio` (if `sounddevice` has issues on your machine)

**Requirements (Windows)**
```bash
pip install sounddevice numpy soundfile faster-whisper   # try this first

# If sounddevice has problems, use pyaudio fallback:
pip install pyaudio
# If pip install pyaudio fails:
pip install pipwin && pipwin install pyaudio
```

**Prerequisites**: run cells 1–2 first so `llm`, `memory`, `agent` are loaded.

In [23]:
# ── Windows Microphone Test ───────────────────────────────────────────────────
import platform
print(f"Platform: {platform.system()} {platform.version()[:60]}")

SAMPLE_RATE = 16000
DURATION    = 4

# ── Try sounddevice first (usually works on Windows without extra setup) ──────
_sd_ok = False
try:
    import sounddevice as sd
    import numpy as np
    _sd_ok = True
    print("Audio backend : sounddevice ✓")
except ImportError:
    print("sounddevice not found — switching to pyaudio fallback ...")

if _sd_ok:
    # ── sounddevice path ──────────────────────────────────────────────────────
    print("\n── Available microphone devices ─────────────────────────────")
    for i, dev in enumerate(sd.query_devices()):
        if dev['max_input_channels'] > 0:
            default = " ◄ default" if i == sd.default.device[0] else ""
            print(f"  [{i:2d}] {dev['name']:<45} ch={dev['max_input_channels']}{default}")
    print()

    DEVICE = None   # set to a specific index from the list above if wrong mic is used

    print(f"Recording {DURATION}s — speak now!  (e.g. 'Cathey, turn on the light.')")
    audio = sd.rec(
        int(DURATION * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype='float32',
        device=DEVICE,
    )
    sd.wait()
    print("Done.\n")

    energy = float(np.mean(np.abs(audio)))
    print(f"Energy level : {energy:.6f}")
    if energy < 0.003:
        print("⚠  Very quiet — check Windows Settings → System → Sound → Input device.")
    else:
        print("✓  Audio captured.")
    audio_arr = audio

else:
    # ── pyaudio fallback path ─────────────────────────────────────────────────
    try:
        import pyaudio
        import numpy as np
        print("Audio backend : pyaudio (fallback) ✓")
    except ImportError:
        print("[ERROR] Neither sounddevice nor pyaudio is installed.")
        print("Install: pip install sounddevice   OR   pip install pyaudio")
        raise

    CHUNK  = 1024
    FORMAT = pyaudio.paInt16

    pa = pyaudio.PyAudio()
    print("\n── Available microphone devices (pyaudio) ───────────────────")
    for i in range(pa.get_device_count()):
        info = pa.get_device_info_by_index(i)
        if info['maxInputChannels'] > 0:
            print(f"  [{i:2d}] {info['name']}")
    print()

    DEVICE_INDEX = None   # None = default; set to index above if needed
    stream = pa.open(
        format=FORMAT, channels=1, rate=SAMPLE_RATE,
        input=True, frames_per_buffer=CHUNK,
        input_device_index=DEVICE_INDEX,
    )
    print(f"Recording {DURATION}s via pyaudio — speak now!")
    frames = [stream.read(CHUNK) for _ in range(int(SAMPLE_RATE / CHUNK * DURATION))]
    stream.stop_stream()
    stream.close()
    pa.terminate()
    print("Done.\n")

    raw = b''.join(frames)
    audio_arr = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    audio_arr = audio_arr.reshape(-1, 1)
    energy = float(np.mean(np.abs(audio_arr)))
    print(f"Energy level : {energy:.6f}")
    if energy < 0.003:
        print("⚠  Very quiet — check Windows Settings → Sound → Recording tab → mic properties.")
    else:
        print("✓  Audio captured.")

# ── Common: STT + Agent ───────────────────────────────────────────────────────
try:
    from audio import STTModel
    stt = STTModel()
    text = stt.transcribe(audio_arr)
    print(f"\nTranscription: {text!r}")
except Exception as e:
    print(f"\n[STT ERROR] {e}")
    text = None

if text:
    print("\n── Agent result ─────────────────────────────────────────────")
    result = agent.handle(text, verbose=True)
    print(f"valid={result['valid']}  reason={result['reason']}  "
          f"exec={result.get('execution')}")

Platform: Darwin Darwin Kernel Version 25.3.0: Wed Jan 28 20:56:42 PST 2026; 
Audio backend : sounddevice ✓

── Available microphone devices ─────────────────────────────
  [ 0] MacBook Pro麦克风                                ch=1 ◄ default
  [ 2] “Huawei Mate XT 非凡大师”的麦克风                     ch=1

Recording 4s — speak now!  (e.g. 'Cathey, turn on the light.')
Done.

Energy level : 0.000938
⚠  Very quiet — check Windows Settings → System → Sound → Input device.
Loading STT (tiny.en) ...
STT ready.

Transcription: ''
